# 🎬 Buscador de clientes para edição de vídeo

Procura negócios no Google Maps que podem comprar seus pacotes de edição de vídeo e monta uma lista
com telefone, Instagram, WhatsApp e uma nota para cada cliente.

**Como usar (é só clicar):**
1. *(Uma vez só)* Clique na chave 🔑 na barra da esquerda → **Adicionar novo secret** →
   nome `GOOGLE_MAPS_API_KEY`, valor = sua chave do Google → ative o acesso do notebook.
   Sem a chave ele usa o OpenStreetMap, que é grátis mas tem menos telefones.
2. Clique no ▶ do **Passo 1**.
3. Preencha a cidade e os nichos no **Passo 2** e clique no ▶. Na primeira vez o Google pede
   permissão para salvar no seu Google Drive: aceite, assim o histórico e a trava de gastos ficam guardados.

A trava de gastos para as buscas no Google em 900 por mês, abaixo da cota grátis de 1.000.

In [ ]:
# @title Passo 1: preparar o programa (só clicar ▶)
# Para mudar a mensagem do WhatsApp, edite /content/mensagem.txt depois deste passo.
from pathlib import Path

ARQUIVOS = {
"buscador_clientes/__init__.py": "\"\"\"Buscador de clientes para pacotes de edição de vídeo.\"\"\"\n\n__version__ = \"1.0.0\"\n",
"buscador_clientes/__main__.py": "from .cli import main\n\nraise SystemExit(main())\n",
"buscador_clientes/analise.py": "\"\"\"Visita o site de cada negócio, descobre redes sociais/contatos e dá uma nota ao lead.\"\"\"\n\nimport re\nimport urllib.parse\nfrom concurrent.futures import ThreadPoolExecutor\n\nfrom .http import ErroHTTP, get_texto\nfrom .nichos import NICHOS\n\nREDES = {\n    \"instagram\": re.compile(r\"https?://(?:www\\.)?instagram\\.com/(?!p/|reel/|explore/|accounts/)([A-Za-z0-9_.]+)\", re.I),\n    \"facebook\": re.compile(r\"https?://(?:www\\.|m\\.)?facebook\\.com/(?!sharer|share|plugins|dialog|tr\\b)([A-Za-z0-9_.\\-]+)\", re.I),\n    \"youtube\": re.compile(r\"https?://(?:www\\.)?youtube\\.com/(?:@|channel/|c/|user/)[A-Za-z0-9_.\\-]+\", re.I),\n    \"tiktok\": re.compile(r\"https?://(?:www\\.)?tiktok\\.com/@[A-Za-z0-9_.]+\", re.I),\n    \"whatsapp\": re.compile(r\"https?://(?:api\\.whatsapp\\.com/send\\?phone=|wa\\.me/)(\\d{10,13})\", re.I),\n}\nEMAIL = re.compile(r\"[A-Za-z0-9._%+\\-]+@[A-Za-z0-9.\\-]+\\.[A-Za-z]{2,}\")\nEMAIL_FALSO = re.compile(r\"\\.(png|jpe?g|gif|webp|svg)$|example\\.|sentry|wixpress|@\\d|seudominio|email\\.com$\", re.I)\n\n\ndef extrair_contatos(html):\n    \"\"\"Procura links de redes sociais, WhatsApp e e-mail dentro do HTML.\"\"\"\n    achados = {}\n    for rede, padrao in REDES.items():\n        m = padrao.search(html)\n        if m:\n            achados[rede] = m.group(1) if rede == \"whatsapp\" else m.group(0)\n    for email in EMAIL.findall(html):\n        if not EMAIL_FALSO.search(email):\n            achados[\"email\"] = email.lower()\n            break\n    return achados\n\n\ndef analisar_site(lead):\n    if not lead.site:\n        return\n    url = lead.site if lead.site.startswith(\"http\") else f\"http://{lead.site}\"\n    host = urllib.parse.urlparse(url).netloc.lower()\n    # Às vezes o \"site\" cadastrado já é a página do Instagram/Facebook.\n    for rede in (\"instagram\", \"facebook\"):\n        if rede in host:\n            if not getattr(lead, rede):\n                setattr(lead, rede, url)\n            return\n    try:\n        html = get_texto(url)\n    except ErroHTTP:\n        return\n    for campo, valor in extrair_contatos(html).items():\n        if not getattr(lead, campo):\n            setattr(lead, campo, valor)\n\n\ndef analisar_sites(leads, threads=10, log=print):\n    com_site = [l for l in leads if l.site]\n    if not com_site:\n        return\n    log(f\"Analisando {len(com_site)} sites para achar Instagram, WhatsApp e e-mail...\")\n    with ThreadPoolExecutor(max_workers=threads) as pool:\n        list(pool.map(analisar_site, com_site))\n\n\ndef pontuar(lead):\n    \"\"\"Nota de 0 a 100: quanto maior, mais chance de o negócio comprar edição de vídeo.\"\"\"\n    pontos, motivos = 0, []\n\n    peso = NICHOS.get(lead.nicho, {}).get(\"peso\", 10)\n    pontos += peso\n    if peso >= 12:\n        motivos.append(\"nicho que usa muito vídeo\")\n\n    if lead.instagram or lead.tiktok or lead.facebook:\n        pontos += 20\n        motivos.append(\"ativo nas redes sociais\")\n    if lead.youtube or lead.tiktok:\n        pontos += 10\n        motivos.append(\"já publica vídeos (pode terceirizar a edição)\")\n    elif lead.instagram:\n        pontos += 10\n        motivos.append(\"tem Instagram mas não tem YouTube/TikTok (oportunidade)\")\n\n    if lead.whatsapp or eh_celular(lead.telefone):\n        pontos += 15\n        motivos.append(\"tem WhatsApp/celular\")\n    elif lead.telefone:\n        pontos += 8\n    if lead.email:\n        pontos += 5\n    if lead.site:\n        pontos += 5\n\n    if lead.avaliacoes:\n        if lead.avaliacoes >= 200:\n            pontos += 15\n            motivos.append(f\"{lead.avaliacoes} avaliações (negócio grande)\")\n        elif lead.avaliacoes >= 50:\n            pontos += 10\n            motivos.append(f\"{lead.avaliacoes} avaliações\")\n        elif lead.avaliacoes >= 10:\n            pontos += 5\n    if lead.nota and lead.nota >= 4.5:\n        pontos += 5\n\n    if not (lead.telefone or lead.whatsapp or lead.email or lead.instagram):\n        pontos = min(pontos, 20)\n        motivos.append(\"sem contato encontrado\")\n\n    lead.pontuacao = min(pontos, 100)\n    lead.motivos = motivos\n\n\ndef so_digitos(texto):\n    return \"\".join(c for c in texto or \"\" if c.isdigit())\n\n\ndef normalizar_telefone_br(telefone):\n    \"\"\"Devolve o telefone no formato 55DDDNUMERO, ou '' se não parecer válido.\"\"\"\n    d = so_digitos(telefone)\n    if d.startswith(\"0\"):\n        d = d.lstrip(\"0\")\n        if len(d) in (12, 13):  # 0 + operadora (ex.: 021) + DDD + número\n            d = d[2:]\n    if d.startswith(\"55\") and len(d) in (12, 13):\n        return d\n    if len(d) in (10, 11):\n        return \"55\" + d\n    return \"\"\n\n\ndef eh_celular(telefone):\n    d = normalizar_telefone_br(telefone)\n    return len(d) == 13 and d[4] == \"9\"\n\n\ndef link_whatsapp(lead, mensagem):\n    numero = normalizar_telefone_br(lead.whatsapp) or (\n        normalizar_telefone_br(lead.telefone) if eh_celular(lead.telefone) else \"\"\n    )\n    if not numero:\n        return \"\"\n    texto = mensagem.replace(\"{nome}\", lead.nome).replace(\"{nicho}\", NICHOS.get(lead.nicho, {}).get(\"busca\", lead.nicho))\n    return f\"https://wa.me/{numero}?text={urllib.parse.quote(texto)}\"\n",
"buscador_clientes/cli.py": "\"\"\"Programa principal: python buscar_clientes.py --cidade \"Campinas, SP\" --nichos academia,restaurante\"\"\"\n\nimport argparse\nimport os\nimport sys\nfrom datetime import datetime\nfrom pathlib import Path\n\nfrom . import exportar, google_places, openstreetmap\nfrom .analise import analisar_sites, link_whatsapp, pontuar\nfrom .http import ErroHTTP\nfrom .nichos import NICHOS, NICHOS_PADRAO, resolver_nichos\n\nPASTA_PROGRAMA = Path(__file__).resolve().parent.parent\nMENSAGEM_PADRAO = (\n    \"Olá, {nome}! Tudo bem? Vi o trabalho de vocês e acho que vídeos bem editados \"\n    \"podem trazer muito mais clientes pelo Instagram e TikTok. Eu trabalho com pacotes \"\n    \"de edição de vídeo (Reels, anúncios e vídeos institucionais). Posso te mandar alguns exemplos?\"\n)\n\n\ndef ler_chave_google(valor_cli):\n    if valor_cli:\n        return valor_cli.strip()\n    if os.environ.get(\"GOOGLE_MAPS_API_KEY\"):\n        return os.environ[\"GOOGLE_MAPS_API_KEY\"].strip()\n    arquivo = PASTA_PROGRAMA / \"chave_google.txt\"\n    if arquivo.exists():\n        return arquivo.read_text(encoding=\"utf-8\").strip()\n    return \"\"\n\n\ndef ler_mensagem(caminho):\n    arquivo = Path(caminho) if caminho else PASTA_PROGRAMA / \"mensagem.txt\"\n    if arquivo.exists():\n        texto = arquivo.read_text(encoding=\"utf-8\").strip()\n        if texto:\n            return texto\n    return MENSAGEM_PADRAO\n\n\ndef perguntar(args):\n    \"\"\"Modo passo a passo para quem não quer digitar parâmetros.\"\"\"\n    print(\"=== Buscador de clientes para edição de vídeo ===\\n\")\n    args.cidade = input('Em qual cidade procurar? (ex.: \"Campinas, SP\" — várias separadas por \";\")\\n> ').strip()\n    print(\"\\nNichos disponíveis:\")\n    print(\"  \" + \", \".join(NICHOS))\n    print(f\"(Enter = {', '.join(NICHOS_PADRAO)}; 'todos' = todos; pode digitar outro termo, ex.: nutricionista)\")\n    args.nichos = input(\"> \").strip()\n    minimo = input(\"\\nPontuação mínima para entrar na lista? (0 a 100, Enter = 0)\\n> \").strip()\n    args.min_pontuacao = int(minimo) if minimo.isdigit() else 0\n    print()\n\n\ndef criar_parser():\n    p = argparse.ArgumentParser(\n        description=\"Encontra negócios no Google Maps / OpenStreetMap que podem comprar pacotes de edição de vídeo.\"\n    )\n    p.add_argument(\"--cidade\", help='Cidade(s), ex.: \"Campinas, SP\". Várias separadas por \";\"')\n    p.add_argument(\"--nichos\", help=f\"Separados por vírgula. Disponíveis: {', '.join(NICHOS)} ou 'todos'\")\n    p.add_argument(\"--fonte\", choices=[\"auto\", \"google\", \"osm\", \"ambas\"], default=\"auto\",\n                   help=\"auto = Google se houver chave, senão OpenStreetMap (padrão)\")\n    p.add_argument(\"--chave-google\", help=\"Chave da Places API (ou use chave_google.txt / GOOGLE_MAPS_API_KEY)\")\n    p.add_argument(\"--max-por-busca\", type=int, default=60, help=\"Máximo de resultados do Google por nicho (até 60)\")\n    p.add_argument(\"--limite-mensal\", type=int, default=google_places.LIMITE_MENSAL_PADRAO,\n                   help=\"Máximo de consultas ao Google por mês, para ficar na cota grátis (padrão: 900)\")\n    p.add_argument(\"--min-pontuacao\", type=int, default=0, help=\"Só exporta leads com pontuação >= este valor\")\n    p.add_argument(\"--sem-analise-site\", action=\"store_true\", help=\"Não visita os sites (mais rápido, menos dados)\")\n    p.add_argument(\"--mensagem\", help=\"Arquivo .txt com a mensagem do WhatsApp (use {nome} para o nome do negócio)\")\n    p.add_argument(\"--saida\", default=str(PASTA_PROGRAMA / \"resultados\"), help=\"Pasta onde salvar os arquivos\")\n    p.add_argument(\"--incluir-repetidos\", action=\"store_true\",\n                   help=\"Inclui negócios que já apareceram em buscas anteriores\")\n    return p\n\n\ndef carregar_historico(pasta):\n    arquivo = pasta / \"historico.txt\"\n    if not arquivo.exists():\n        return set()\n    return set(arquivo.read_text(encoding=\"utf-8\").splitlines())\n\n\ndef salvar_historico(pasta, leads):\n    with open(pasta / \"historico.txt\", \"a\", encoding=\"utf-8\") as f:\n        for lead in leads:\n            f.write(\"|\".join(lead.chave()) + \"\\n\")\n\n\ndef coletar(args, cidades, nichos, chave, pasta):\n    usar_google = args.fonte in (\"google\", \"ambas\") or (args.fonte == \"auto\" and chave)\n    usar_osm = args.fonte in (\"osm\", \"ambas\") or (args.fonte == \"auto\" and not chave)\n    if usar_google and not chave:\n        sys.exit(\"Para usar o Google Maps é preciso uma chave de API. Veja o README (seção 'Chave do Google').\")\n    if args.fonte == \"auto\" and not chave:\n        print(\"Sem chave do Google: usando o OpenStreetMap (grátis). Para mais resultados, veja o README.\\n\")\n\n    contador = google_places.ContadorUso(pasta / \"uso_google.json\", args.limite_mensal)\n    leads = []\n    for cidade in cidades:\n        print(f\"Procurando em {cidade}...\")\n        if usar_google:\n            for n in nichos:\n                try:\n                    leads += google_places.buscar(chave, NICHOS[n][\"busca\"], cidade, n, args.max_por_busca,\n                                                  contador=contador)\n                except ErroHTTP as e:\n                    print(f\"  Erro no Google Maps ({n}): {e}\")\n                except google_places.LimiteMensalAtingido as e:\n                    print(f\"  {e}\")\n                    usar_google = False\n                    break\n        if usar_osm:\n            try:\n                leads += openstreetmap.buscar(cidade, nichos)\n            except ErroHTTP as e:\n                print(f\"  Erro no OpenStreetMap: {e}\")\n    if chave and args.fonte != \"osm\":\n        print(f\"Consultas ao Google neste mês: {contador.usados} de {contador.limite} (cota grátis).\\n\")\n    return leads\n\n\ndef main(argv=None):\n    args = criar_parser().parse_args(argv)\n    if not args.cidade:\n        perguntar(args)\n    cidades = [c.strip() for c in (args.cidade or \"\").split(\";\") if c.strip()]\n    if not cidades:\n        print(\"Informe pelo menos uma cidade.\")\n        return 1\n    nichos = resolver_nichos((args.nichos or \"\").split(\",\") if args.nichos else [])\n    chave = ler_chave_google(args.chave_google)\n\n    pasta = Path(args.saida)\n    pasta.mkdir(parents=True, exist_ok=True)\n    leads = coletar(args, cidades, nichos, chave, pasta)\n\n    unicos = {}\n    for lead in leads:\n        unicos.setdefault(lead.chave(), lead)\n    leads = [l for l in unicos.values() if l.nome]\n\n    if not args.incluir_repetidos:\n        historico = carregar_historico(pasta)\n        antes = len(leads)\n        leads = [l for l in leads if \"|\".join(l.chave()) not in historico]\n        if antes - len(leads):\n            print(f\"{antes - len(leads)} já tinham aparecido em buscas anteriores e foram ignorados \"\n                  \"(use --incluir-repetidos para mostrar).\")\n\n    if not leads:\n        print(\"Nenhum cliente novo encontrado. Tente outra cidade, outros nichos ou --fonte ambas.\")\n        return 0\n\n    if not args.sem_analise_site:\n        analisar_sites(leads)\n\n    mensagem = ler_mensagem(args.mensagem)\n    for lead in leads:\n        pontuar(lead)\n        lead.link_whatsapp = link_whatsapp(lead, mensagem)\n    leads = [l for l in leads if l.pontuacao >= args.min_pontuacao]\n    leads.sort(key=lambda l: (-l.pontuacao, -(l.avaliacoes or 0)))\n    if not leads:\n        print(\"Nenhum cliente passou da pontuação mínima. Diminua --min-pontuacao.\")\n        return 0\n\n    nome_base = f\"clientes_{datetime.now():%Y-%m-%d_%H%M}\"\n    salvar_historico(pasta, leads)\n    exportar.salvar_csv(leads, pasta / f\"{nome_base}.csv\")\n    arquivos = [f\"{nome_base}.csv\"]\n    if exportar.salvar_xlsx(leads, pasta / f\"{nome_base}.xlsx\"):\n        arquivos.append(f\"{nome_base}.xlsx\")\n    exportar.salvar_html(leads, pasta / f\"{nome_base}.html\", f\"Clientes em {', '.join(cidades)}\")\n    arquivos.append(f\"{nome_base}.html\")\n\n    com_zap = sum(1 for l in leads if l.link_whatsapp)\n    print(f\"\\nPronto! {len(leads)} clientes encontrados ({com_zap} com link direto para o WhatsApp).\")\n    print(\"Melhores leads:\")\n    for l in leads[:10]:\n        print(f\"  [{l.pontuacao:3d}] {l.nome} — {l.telefone or l.instagram or l.email or 'sem contato'}\")\n    print(f\"\\nArquivos salvos em {pasta}:\")\n    for a in arquivos:\n        print(f\"  - {a}\")\n    return 0\n",
"buscador_clientes/exportar.py": "\"\"\"Salva a lista de clientes em CSV (abre no Excel), Excel (.xlsx) e uma página HTML.\"\"\"\n\nimport csv\nimport html\nfrom pathlib import Path\n\nfrom .nichos import NICHOS\n\nCOLUNAS = [\n    (\"pontuacao\", \"Pontuação\"),\n    (\"nome\", \"Nome\"),\n    (\"nicho\", \"Nicho\"),\n    (\"telefone\", \"Telefone\"),\n    (\"link_whatsapp\", \"Chamar no WhatsApp\"),\n    (\"instagram\", \"Instagram\"),\n    (\"email\", \"E-mail\"),\n    (\"site\", \"Site\"),\n    (\"facebook\", \"Facebook\"),\n    (\"youtube\", \"YouTube\"),\n    (\"tiktok\", \"TikTok\"),\n    (\"nota\", \"Nota\"),\n    (\"avaliacoes\", \"Avaliações\"),\n    (\"endereco\", \"Endereço\"),\n    (\"cidade\", \"Cidade\"),\n    (\"link_mapa\", \"Ver no mapa\"),\n    (\"motivos\", \"Por que é um bom lead\"),\n    (\"fonte\", \"Fonte\"),\n]\nLINKS = {\"link_whatsapp\", \"instagram\", \"site\", \"facebook\", \"youtube\", \"tiktok\", \"link_mapa\"}\n\n\ndef _valor(lead, campo):\n    v = getattr(lead, campo)\n    if campo == \"motivos\":\n        return \"; \".join(v)\n    if campo == \"nicho\":\n        return NICHOS.get(v, {}).get(\"busca\", v)\n    return \"\" if v is None else v\n\n\ndef salvar_csv(leads, caminho):\n    # \";\" e utf-8-sig fazem o Excel em português abrir o arquivo com acentos e colunas certas.\n    with open(caminho, \"w\", newline=\"\", encoding=\"utf-8-sig\") as f:\n        w = csv.writer(f, delimiter=\";\")\n        w.writerow([titulo for _, titulo in COLUNAS])\n        for lead in leads:\n            w.writerow([_valor(lead, c) for c, _ in COLUNAS])\n\n\ndef salvar_xlsx(leads, caminho):\n    \"\"\"Gera a planilha Excel se a biblioteca openpyxl estiver instalada. Devolve True se salvou.\"\"\"\n    try:\n        from openpyxl import Workbook\n        from openpyxl.styles import Font, PatternFill\n    except ImportError:\n        return False\n    wb = Workbook()\n    ws = wb.active\n    ws.title = \"Clientes\"\n    ws.append([titulo for _, titulo in COLUNAS])\n    for cel in ws[1]:\n        cel.font = Font(bold=True, color=\"FFFFFF\")\n        cel.fill = PatternFill(\"solid\", fgColor=\"4F46E5\")\n    for lead in leads:\n        ws.append([_valor(lead, c) for c, _ in COLUNAS])\n        linha = ws.max_row\n        for i, (campo, _) in enumerate(COLUNAS, start=1):\n            url = getattr(lead, campo)\n            if campo in LINKS and isinstance(url, str) and url.startswith(\"http\"):\n                cel = ws.cell(row=linha, column=i)\n                cel.hyperlink = url\n                cel.value = \"Abrir conversa\" if campo == \"link_whatsapp\" else url\n                cel.font = Font(color=\"0563C1\", underline=\"single\")\n    larguras = {\"Nome\": 35, \"Endereço\": 45, \"Por que é um bom lead\": 60, \"Chamar no WhatsApp\": 18}\n    for i, (_, titulo) in enumerate(COLUNAS, start=1):\n        ws.column_dimensions[ws.cell(row=1, column=i).column_letter].width = larguras.get(titulo, 20)\n    ws.freeze_panes = \"C2\"\n    ws.auto_filter.ref = ws.dimensions\n    wb.save(caminho)\n    return True\n\n\ndef _link(url, texto=None):\n    if not url:\n        return \"\"\n    if not url.startswith(\"http\"):\n        url = \"https://\" + url\n    return f'<a href=\"{html.escape(url)}\" target=\"_blank\" rel=\"noopener\">{html.escape(texto or url)}</a>'\n\n\ndef salvar_html(leads, caminho, titulo):\n    linhas = []\n    for l in leads:\n        cor = \"alta\" if l.pontuacao >= 60 else \"media\" if l.pontuacao >= 40 else \"baixa\"\n        redes = \" \".join(\n            filter(None, [\n                _link(l.instagram, \"Instagram\"), _link(l.facebook, \"Facebook\"),\n                _link(l.youtube, \"YouTube\"), _link(l.tiktok, \"TikTok\"), _link(l.site, \"Site\"),\n            ])\n        )\n        zap = f'<a class=\"zap\" href=\"{html.escape(l.link_whatsapp)}\" target=\"_blank\">WhatsApp</a>' if l.link_whatsapp else \"\"\n        nota = f\"{l.nota:.1f} ★ ({l.avaliacoes or 0})\" if l.nota else \"\"\n        linhas.append(\n            f\"<tr><td><span class='nota {cor}'>{l.pontuacao}</span></td>\"\n            f\"<td><strong>{html.escape(l.nome)}</strong><br><small>{html.escape(_valor(l, 'nicho'))}</small></td>\"\n            f\"<td>{html.escape(l.telefone)}<br>{zap}</td>\"\n            f\"<td>{redes}<br><small>{html.escape(l.email)}</small></td>\"\n            f\"<td>{nota}</td>\"\n            f\"<td><small>{html.escape(l.endereco)}</small><br>{_link(l.link_mapa, 'Ver no mapa')}</td>\"\n            f\"<td><small>{html.escape('; '.join(l.motivos))}</small></td></tr>\"\n        )\n    pagina = f\"\"\"<!doctype html>\n<html lang=\"pt-BR\"><head><meta charset=\"utf-8\"><meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">\n<title>{html.escape(titulo)}</title>\n<style>\nbody{{font-family:system-ui,sans-serif;margin:0;padding:16px;background:#f6f7fb;color:#1f2330}}\nh1{{font-size:1.3rem}} input{{padding:8px;width:100%;max-width:400px;margin-bottom:12px;border:1px solid #ccc;border-radius:6px}}\ntd:nth-child(3){{white-space:nowrap}} .tabela{{overflow-x:auto}} table{{border-collapse:collapse;width:100%;background:#fff}}\nth,td{{padding:8px;border-bottom:1px solid #e5e7eb;text-align:left;vertical-align:top;font-size:.9rem}}\nth{{background:#4f46e5;color:#fff;position:sticky;top:0}} a{{color:#4338ca;margin-right:6px}}\n.nota{{display:inline-block;min-width:32px;text-align:center;padding:4px;border-radius:6px;font-weight:bold;color:#fff}}\n.alta{{background:#16a34a}} .media{{background:#d97706}} .baixa{{background:#9ca3af}}\n.zap{{display:inline-block;margin-top:4px;background:#25d366;color:#fff;padding:3px 8px;border-radius:4px;text-decoration:none}}\n</style></head><body>\n<h1>{html.escape(titulo)} — {len(leads)} clientes</h1>\n<input id=\"filtro\" placeholder=\"Filtrar por nome, nicho, bairro...\" oninput=\"filtrar(this.value)\">\n<div class=\"tabela\"><table><thead><tr><th>Nota</th><th>Negócio</th><th>Telefone</th><th>Redes / e-mail</th>\n<th>Google</th><th>Endereço</th><th>Por que é um bom lead</th></tr></thead><tbody>\n{''.join(linhas)}\n</tbody></table></div>\n<script>function filtrar(t){{t=t.toLowerCase();document.querySelectorAll('tbody tr').forEach(r=>{{r.style.display=r.innerText.toLowerCase().includes(t)?'':'none'}})}}</script>\n</body></html>\"\"\"\n    Path(caminho).write_text(pagina, encoding=\"utf-8\")\n",
"buscador_clientes/google_places.py": "\"\"\"Busca no Google Maps pela API oficial (Places API - New).\n\nPrecisa de uma chave de API do Google Cloud. O Google dá um crédito mensal\ngratuito que cobre milhares de buscas; veja o README para criar a chave.\n\"\"\"\n\nimport json\nimport time\nfrom datetime import datetime\nfrom pathlib import Path\n\nfrom .http import post_json\nfrom .modelo import Lead\n\nURL = \"https://places.googleapis.com/v1/places:searchText\"\nCAMPOS = \",\".join(\n    [\n        \"places.displayName\",\n        \"places.formattedAddress\",\n        \"places.nationalPhoneNumber\",\n        \"places.internationalPhoneNumber\",\n        \"places.websiteUri\",\n        \"places.rating\",\n        \"places.userRatingCount\",\n        \"places.googleMapsUri\",\n        \"places.businessStatus\",\n        \"nextPageToken\",\n    ]\n)\n\n\n# O Google dá 1.000 consultas grátis por mês para buscas que trazem telefone, site e\n# avaliações. Paramos antes disso, com folga, para a conta nunca ser cobrada.\nLIMITE_MENSAL_PADRAO = 900\n\n\nclass LimiteMensalAtingido(Exception):\n    pass\n\n\nclass ContadorUso:\n    \"\"\"Conta as consultas ao Google no mês e bloqueia quando chega no limite.\"\"\"\n\n    def __init__(self, arquivo, limite):\n        self.arquivo = Path(arquivo)\n        self.limite = limite\n        try:\n            self.dados = json.loads(self.arquivo.read_text(encoding=\"utf-8\"))\n        except (OSError, ValueError):\n            self.dados = {}\n\n    @property\n    def mes(self):\n        return f\"{datetime.now():%Y-%m}\"\n\n    @property\n    def usados(self):\n        return self.dados.get(self.mes, 0)\n\n    def registrar(self):\n        if self.usados >= self.limite:\n            raise LimiteMensalAtingido(\n                f\"Limite de {self.limite} consultas ao Google neste mês atingido. \"\n                \"Para não gastar nada, as buscas no Google voltam no mês que vem \"\n                \"(o OpenStreetMap continua grátis: use --fonte osm).\"\n            )\n        self.dados[self.mes] = self.usados + 1\n        self.arquivo.parent.mkdir(parents=True, exist_ok=True)\n        self.arquivo.write_text(json.dumps(self.dados), encoding=\"utf-8\")\n\n\ndef buscar(chave_api, termo, cidade, nicho, max_resultados=60, log=print, contador=None):\n    \"\"\"Devolve até `max_resultados` (o Google limita a 60) leads para \"termo em cidade\".\"\"\"\n    leads = []\n    corpo = {\"textQuery\": f\"{termo} em {cidade}\", \"languageCode\": \"pt-BR\", \"regionCode\": \"BR\", \"pageSize\": 20}\n    cabecalhos = {\"X-Goog-Api-Key\": chave_api, \"X-Goog-FieldMask\": CAMPOS}\n    while len(leads) < max_resultados:\n        if contador:\n            contador.registrar()\n        resposta = post_json(URL, corpo, cabecalhos)\n        for lugar in resposta.get(\"places\", []):\n            if lugar.get(\"businessStatus\") not in (None, \"OPERATIONAL\"):\n                continue  # fechado temporária ou definitivamente\n            leads.append(\n                Lead(\n                    nome=lugar.get(\"displayName\", {}).get(\"text\", \"\"),\n                    nicho=nicho,\n                    fonte=\"Google Maps\",\n                    endereco=lugar.get(\"formattedAddress\", \"\"),\n                    cidade=cidade,\n                    telefone=lugar.get(\"nationalPhoneNumber\") or lugar.get(\"internationalPhoneNumber\", \"\"),\n                    site=lugar.get(\"websiteUri\", \"\"),\n                    nota=lugar.get(\"rating\"),\n                    avaliacoes=lugar.get(\"userRatingCount\"),\n                    link_mapa=lugar.get(\"googleMapsUri\", \"\"),\n                )\n            )\n        token = resposta.get(\"nextPageToken\")\n        if not token:\n            break\n        corpo[\"pageToken\"] = token\n        time.sleep(1)  # o token da próxima página demora um instante para valer\n    log(f\"  Google Maps: {len(leads)} encontrados para '{termo}'\")\n    return leads[:max_resultados]\n",
"buscador_clientes/http.py": "\"\"\"Pequeno cliente HTTP usando apenas a biblioteca padrão (sem instalar nada).\"\"\"\n\nimport json\nimport urllib.error\nimport urllib.parse\nimport urllib.request\n\nUSER_AGENT = \"BuscadorClientes/1.0 (prospeccao de clientes para edicao de video)\"\n\n\nclass ErroHTTP(Exception):\n    pass\n\n\ndef _abrir(req, timeout):\n    try:\n        with urllib.request.urlopen(req, timeout=timeout) as resp:\n            return resp.read(), resp.headers.get_content_charset() or \"utf-8\"\n    except urllib.error.HTTPError as e:\n        corpo = e.read().decode(\"utf-8\", \"replace\")[:500]\n        raise ErroHTTP(f\"HTTP {e.code} em {req.full_url}: {corpo}\") from e\n    except (urllib.error.URLError, TimeoutError, OSError) as e:\n        raise ErroHTTP(f\"Falha ao acessar {req.full_url}: {e}\") from e\n\n\ndef get_json(url, params=None, headers=None, timeout=30):\n    if params:\n        url = f\"{url}?{urllib.parse.urlencode(params)}\"\n    req = urllib.request.Request(url, headers={\"User-Agent\": USER_AGENT, **(headers or {})})\n    dados, charset = _abrir(req, timeout)\n    return json.loads(dados.decode(charset, \"replace\"))\n\n\ndef post_json(url, corpo, headers=None, timeout=30):\n    req = urllib.request.Request(\n        url,\n        data=json.dumps(corpo).encode(\"utf-8\"),\n        headers={\"User-Agent\": USER_AGENT, \"Content-Type\": \"application/json\", **(headers or {})},\n        method=\"POST\",\n    )\n    dados, charset = _abrir(req, timeout)\n    return json.loads(dados.decode(charset, \"replace\"))\n\n\ndef post_form(url, campos, timeout=120):\n    req = urllib.request.Request(\n        url,\n        data=urllib.parse.urlencode(campos).encode(\"utf-8\"),\n        headers={\"User-Agent\": USER_AGENT},\n        method=\"POST\",\n    )\n    dados, charset = _abrir(req, timeout)\n    return json.loads(dados.decode(charset, \"replace\"))\n\n\ndef get_texto(url, timeout=10, limite_bytes=600_000):\n    \"\"\"Baixa uma página (até `limite_bytes`) e devolve o HTML como texto.\"\"\"\n    req = urllib.request.Request(\n        url,\n        headers={\"User-Agent\": \"Mozilla/5.0 (Windows NT 10.0; Win64; x64)\", \"Accept\": \"text/html\"},\n    )\n    try:\n        with urllib.request.urlopen(req, timeout=timeout) as resp:\n            dados = resp.read(limite_bytes)\n            charset = resp.headers.get_content_charset() or \"utf-8\"\n    except Exception as e:  # sites de terceiros falham de todo jeito; não derruba a busca\n        raise ErroHTTP(f\"Falha ao acessar {url}: {e}\") from e\n    return dados.decode(charset, \"replace\")\n",
"buscador_clientes/modelo.py": "from dataclasses import dataclass, field\n\n\n@dataclass\nclass Lead:\n    nome: str\n    nicho: str\n    fonte: str\n    endereco: str = \"\"\n    cidade: str = \"\"\n    telefone: str = \"\"\n    site: str = \"\"\n    email: str = \"\"\n    instagram: str = \"\"\n    facebook: str = \"\"\n    youtube: str = \"\"\n    tiktok: str = \"\"\n    whatsapp: str = \"\"\n    nota: float | None = None\n    avaliacoes: int | None = None\n    link_mapa: str = \"\"\n    pontuacao: int = 0\n    motivos: list[str] = field(default_factory=list)\n    link_whatsapp: str = \"\"\n\n    def chave(self):\n        \"\"\"Identifica o mesmo negócio vindo de buscas diferentes.\"\"\"\n        digitos = \"\".join(c for c in self.telefone if c.isdigit())[-8:]\n        return (self.nome.strip().lower(), digitos or self.endereco.strip().lower())\n",
"buscador_clientes/nichos.py": "\"\"\"Tipos de negócio que costumam contratar edição de vídeo.\n\nCada nicho tem:\n  - busca:  o termo usado no Google Maps (\"academia em Campinas\")\n  - osm:    as etiquetas equivalentes no OpenStreetMap (chave, valor)\n  - peso:   o quanto esse tipo de negócio costuma investir em vídeo (0 a 15),\n            usado na pontuação do lead\n\"\"\"\n\nNICHOS = {\n    \"academia\": {\"busca\": \"academia\", \"osm\": [(\"leisure\", \"fitness_centre\"), (\"leisure\", \"sports_centre\")], \"peso\": 15},\n    \"personal\": {\"busca\": \"personal trainer\", \"osm\": [], \"peso\": 15},\n    \"estetica\": {\"busca\": \"clínica de estética\", \"osm\": [(\"shop\", \"beauty\")], \"peso\": 15},\n    \"salao\": {\"busca\": \"salão de beleza\", \"osm\": [(\"shop\", \"hairdresser\")], \"peso\": 12},\n    \"barbearia\": {\"busca\": \"barbearia\", \"osm\": [(\"shop\", \"hairdresser\")], \"peso\": 12},\n    \"imobiliaria\": {\"busca\": \"imobiliária\", \"osm\": [(\"office\", \"estate_agent\")], \"peso\": 15},\n    \"restaurante\": {\"busca\": \"restaurante\", \"osm\": [(\"amenity\", \"restaurant\")], \"peso\": 10},\n    \"hamburgueria\": {\"busca\": \"hamburgueria\", \"osm\": [(\"amenity\", \"fast_food\")], \"peso\": 10},\n    \"cafeteria\": {\"busca\": \"cafeteria\", \"osm\": [(\"amenity\", \"cafe\")], \"peso\": 8},\n    \"bar\": {\"busca\": \"bar\", \"osm\": [(\"amenity\", \"bar\"), (\"amenity\", \"pub\")], \"peso\": 8},\n    \"dentista\": {\"busca\": \"dentista\", \"osm\": [(\"amenity\", \"dentist\"), (\"healthcare\", \"dentist\")], \"peso\": 12},\n    \"clinica\": {\"busca\": \"clínica médica\", \"osm\": [(\"amenity\", \"clinic\")], \"peso\": 10},\n    \"advogado\": {\"busca\": \"escritório de advocacia\", \"osm\": [(\"office\", \"lawyer\")], \"peso\": 8},\n    \"loja_roupas\": {\"busca\": \"loja de roupas\", \"osm\": [(\"shop\", \"clothes\"), (\"shop\", \"boutique\")], \"peso\": 12},\n    \"concessionaria\": {\"busca\": \"concessionária de veículos\", \"osm\": [(\"shop\", \"car\")], \"peso\": 12},\n    \"escola\": {\"busca\": \"escola particular\", \"osm\": [(\"amenity\", \"school\")], \"peso\": 8},\n    \"curso\": {\"busca\": \"escola de idiomas\", \"osm\": [(\"amenity\", \"language_school\"), (\"amenity\", \"college\")], \"peso\": 10},\n    \"autoescola\": {\"busca\": \"autoescola\", \"osm\": [(\"amenity\", \"driving_school\")], \"peso\": 8},\n    \"petshop\": {\"busca\": \"pet shop\", \"osm\": [(\"shop\", \"pet\")], \"peso\": 8},\n    \"hotel\": {\"busca\": \"hotel pousada\", \"osm\": [(\"tourism\", \"hotel\"), (\"tourism\", \"guest_house\")], \"peso\": 12},\n    \"eventos\": {\"busca\": \"espaço de eventos buffet\", \"osm\": [(\"amenity\", \"events_venue\")], \"peso\": 12},\n    \"fotografo\": {\"busca\": \"fotógrafo\", \"osm\": [(\"craft\", \"photographer\"), (\"shop\", \"photo\")], \"peso\": 12},\n    \"igreja\": {\"busca\": \"igreja\", \"osm\": [(\"amenity\", \"place_of_worship\")], \"peso\": 8},\n    \"tatuagem\": {\"busca\": \"estúdio de tatuagem\", \"osm\": [(\"shop\", \"tattoo\")], \"peso\": 10},\n    \"agencia\": {\"busca\": \"agência de marketing digital\", \"osm\": [(\"office\", \"advertising_agency\")], \"peso\": 15},\n    \"construtora\": {\"busca\": \"construtora\", \"osm\": [(\"office\", \"construction_company\")], \"peso\": 10},\n    \"padaria\": {\"busca\": \"padaria confeitaria\", \"osm\": [(\"shop\", \"bakery\"), (\"shop\", \"pastry\")], \"peso\": 6},\n}\n\n# Usados quando a pessoa não escolhe nenhum nicho.\nNICHOS_PADRAO = [\"academia\", \"estetica\", \"imobiliaria\", \"restaurante\", \"barbearia\", \"loja_roupas\", \"dentista\"]\n\n\ndef resolver_nichos(nomes):\n    \"\"\"Converte uma lista de nomes (ou 'todos') em chaves válidas de NICHOS.\n\n    Um termo desconhecido vira uma busca livre: assim dá para procurar\n    qualquer coisa no Google Maps, por exemplo \"nutricionista\".\n    \"\"\"\n    if not nomes:\n        return list(NICHOS_PADRAO)\n    if any(n.strip().lower() == \"todos\" for n in nomes):\n        return list(NICHOS)\n    resultado = []\n    for nome in nomes:\n        chave = nome.strip().lower()\n        if not chave:\n            continue\n        if chave not in NICHOS:\n            NICHOS[chave] = {\"busca\": nome.strip(), \"osm\": [], \"peso\": 10}\n        if chave not in resultado:\n            resultado.append(chave)\n    return resultado\n",
"buscador_clientes/openstreetmap.py": "\"\"\"Busca gratuita no OpenStreetMap (não precisa de chave nem cadastro).\n\nTem menos telefones e sites que o Google Maps, mas é 100% grátis.\n\"\"\"\n\nimport time\n\nfrom .http import get_json, post_form\nfrom .modelo import Lead\nfrom .nichos import NICHOS\n\nNOMINATIM = \"https://nominatim.openstreetmap.org/search\"\nOVERPASS = \"https://overpass-api.de/api/interpreter\"\n\n\ndef localizar_cidade(cidade):\n    \"\"\"Devolve o id de área do Overpass para a cidade (ou None).\"\"\"\n    resultados = get_json(\n        NOMINATIM,\n        {\"q\": cidade, \"format\": \"json\", \"limit\": 5, \"countrycodes\": \"br\", \"accept-language\": \"pt-BR\"},\n    )\n    for r in resultados:\n        if r.get(\"osm_type\") == \"relation\":\n            return 3_600_000_000 + int(r[\"osm_id\"])\n    return None\n\n\ndef montar_consulta(area_id, nichos):\n    filtros = sorted({(k, v) for n in nichos for (k, v) in NICHOS[n][\"osm\"]})\n    linhas = [f'  nwr[\"{k}\"=\"{v}\"][\"name\"](area.cidade);' for k, v in filtros]\n    return f\"[out:json][timeout:120];\\narea({area_id})->.cidade;\\n(\\n\" + \"\\n\".join(linhas) + \"\\n);\\nout center tags;\"\n\n\ndef _nicho_do_elemento(tags, nichos):\n    for n in nichos:\n        for k, v in NICHOS[n][\"osm\"]:\n            if tags.get(k) == v:\n                if n == \"barbearia\" and \"salao\" in nichos and tags.get(\"hairdresser\") != \"barber\":\n                    continue\n                return n\n    return nichos[0]\n\n\ndef _endereco(tags):\n    partes = [tags.get(\"addr:street\", \"\"), tags.get(\"addr:housenumber\", \"\"), tags.get(\"addr:suburb\", \"\")]\n    return \", \".join(p for p in partes if p)\n\n\ndef buscar(cidade, nichos, log=print):\n    nichos_osm = [n for n in nichos if NICHOS[n][\"osm\"]]\n    sem_osm = [n for n in nichos if not NICHOS[n][\"osm\"]]\n    if sem_osm:\n        log(f\"  OpenStreetMap não tem categoria para: {', '.join(sem_osm)} (use o Google Maps para esses)\")\n    if not nichos_osm:\n        return []\n\n    area_id = localizar_cidade(cidade)\n    if area_id is None:\n        log(f\"  Não encontrei a cidade '{cidade}' no OpenStreetMap. Tente 'Cidade, UF'.\")\n        return []\n    time.sleep(1)  # política de uso do Nominatim: no máximo 1 requisição por segundo\n\n    dados = post_form(OVERPASS, {\"data\": montar_consulta(area_id, nichos_osm)})\n    leads = []\n    for el in dados.get(\"elements\", []):\n        tags = el.get(\"tags\", {})\n        lat = el.get(\"lat\") or el.get(\"center\", {}).get(\"lat\")\n        lon = el.get(\"lon\") or el.get(\"center\", {}).get(\"lon\")\n        leads.append(\n            Lead(\n                nome=tags.get(\"name\", \"\"),\n                nicho=_nicho_do_elemento(tags, nichos_osm),\n                fonte=\"OpenStreetMap\",\n                endereco=_endereco(tags),\n                cidade=cidade,\n                telefone=tags.get(\"phone\") or tags.get(\"contact:phone\") or tags.get(\"mobile\", \"\"),\n                site=tags.get(\"website\") or tags.get(\"contact:website\", \"\"),\n                email=tags.get(\"email\") or tags.get(\"contact:email\", \"\"),\n                instagram=tags.get(\"contact:instagram\", \"\"),\n                facebook=tags.get(\"contact:facebook\", \"\"),\n                whatsapp=tags.get(\"contact:whatsapp\", \"\"),\n                link_mapa=f\"https://www.google.com/maps/search/?api=1&query={lat},{lon}\" if lat and lon else \"\",\n            )\n        )\n    log(f\"  OpenStreetMap: {len(leads)} encontrados\")\n    return leads\n",
"mensagem.txt": "Olá, {nome}! Tudo bem? Vi o trabalho de vocês e acho que vídeos bem editados podem trazer muito mais clientes pelo Instagram e TikTok. Eu trabalho com pacotes de edição de vídeo (Reels, anúncios e vídeos institucionais). Posso te mandar alguns exemplos?\n"
}

for caminho, conteudo in ARQUIVOS.items():
    Path(caminho).parent.mkdir(parents=True, exist_ok=True)
    Path(caminho).write_text(conteudo, encoding="utf-8")
print("Programa pronto! Agora vá para o Passo 2.")

In [ ]:
# @title Passo 2: buscar clientes
cidade = "Campinas, SP"  # @param {type:"string"}
nichos = "academia, estetica, imobiliaria, restaurante, barbearia"  # @param {type:"string"}
pontuacao_minima = 0  # @param {type:"slider", min:0, max:100, step:5}
fonte = "auto"  # @param ["auto", "google", "osm", "ambas"]
salvar_no_google_drive = True  # @param {type:"boolean"}
# Nichos prontos: academia, personal, estetica, salao, barbearia, imobiliaria, restaurante,
# hamburgueria, cafeteria, bar, dentista, clinica, advogado, loja_roupas, concessionaria, escola,
# curso, autoescola, petshop, hotel, eventos, fotografo, igreja, tatuagem, agencia, construtora,
# padaria, todos — ou qualquer outro termo (ex.: nutricionista).
# Várias cidades: separe com ";" (ex.: "Campinas, SP; Valinhos, SP").

import os
from pathlib import Path

from IPython.display import HTML, display

try:
    from google.colab import userdata
    os.environ["GOOGLE_MAPS_API_KEY"] = userdata.get("GOOGLE_MAPS_API_KEY") or ""
except Exception:
    os.environ.pop("GOOGLE_MAPS_API_KEY", None)
    print("Chave do Google não encontrada nos secrets (🔑): usando o OpenStreetMap.\n")

pasta = Path("/content/resultados")
if salvar_no_google_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    pasta = Path("/content/drive/MyDrive/Buscador de Clientes")

from buscador_clientes.cli import main

main(["--cidade", cidade, "--nichos", nichos, "--fonte", fonte,
      "--min-pontuacao", str(pontuacao_minima), "--saida", str(pasta)])

paginas = sorted(pasta.glob("clientes_*.html"), key=lambda p: p.stat().st_mtime)
if paginas:
    display(HTML(paginas[-1].read_text(encoding="utf-8")))
    planilha = paginas[-1].with_suffix(".xlsx")
    if planilha.exists():
        from google.colab import files
        files.download(str(planilha))